# Advertising Sales Prediction using Multiple Linear Regression
### End-to-End Predictive Modeling, Visual Analysis & Budget Optimization

---

## Project Overview & Business Scenario
A consumer goods company promotes its products across three primary marketing channels:
1. **Television (TV)**
2. **Radio**
3. **Newspaper**

The leadership team wants to understand:
- **Question 1:** How do advertising expenditures across TV, Radio, and Newspaper jointly drive product sales, and which channel is the most cost-effective?
- **Question 2:** What is the expected sales volume if the company invests a planned budget of **TV = $150k**, **Radio = $20k**, and **Newspaper = $30k**?

In this project, we analyze the dataset, fit a **Multiple Linear Regression** model, evaluate its accuracy, present the 3 key visualizations, and deliver actionable marketing recommendations.

## Task Breakdown & Scoring Rubric
This notebook covers all 11 required assignment tasks in logical order:
- **Task 1:** Load and inspect dataset structure.
- **Task 2:** Select input features (`TV`, `Radio`, `Newspaper`) and target variable (`Sales`).
- **Task 3:** Data quality assessment and Exploratory Data Analysis (**Graph 1: Spend vs. Sales**).
- **Task 4:** Train-test split (80% train / 20% test with `random_state=42`).
- **Task 5:** Train the Multiple Linear Regression model.
- **Task 6:** Predict sales on the hold-out test set.
- **Task 7:** Forecast sales for planned budget: TV = 150, Radio = 20, Newspaper = 30.
- **Task 8:** Model evaluation using MAE, MSE, RMSE, and $R^2$ score.
- **Task 9:** Interpret regression coefficients (intercept, strongest and weakest channels).
- **Task 10:** Model evaluation visuals (**Graph 2: Actual vs. Predicted Scatter Plot** & **Graph 3: Sorted Actual vs. Predicted Comparison Bar Chart**).
- **Task 11:** Practical business recommendations and technical improvements.

## Step 1: Import Required Libraries
We import standard Python libraries for data processing, regression modeling, and visualization.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries successfully loaded.")

Libraries successfully loaded.


## Step 2: Load and Inspect the Dataset (Task 1)
We load the advertising dataset and examine its initial rows, shape, and column names.

In [2]:
# Load dataset (supports both uppercase and lowercase filenames)
try:
    df = pd.read_csv('advertising.csv')
except FileNotFoundError:
    df = pd.read_csv('Advertising.csv')

print("First 5 Rows of the Dataset:")
print(df.head())

print(f"\nDataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
print("Columns:", df.columns.tolist())

First 5 Rows of the Dataset:
      TV  Radio  Newspaper  Sales
0  230.1   37.8       69.2   22.1
1   44.5   39.3       45.1   10.4
2   17.2   45.9       69.3   12.0
3  151.5   41.3       58.5   16.5
4  180.8   10.8       58.4   17.9

Dataset Dimensions: 200 rows, 4 columns
Columns: ['TV', 'Radio', 'Newspaper', 'Sales']


## Step 3: Select Features and Target Variable (Task 2)
We separate the input features from the target variable:
- **Features ($X$):** `TV`, `Radio`, and `Newspaper` (advertising spend in thousands of dollars, \$k)
- **Target ($y$):** `Sales` (sales volume in thousands of units)

In [3]:
X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']

print("Features (X) Head:")
print(X.head())

print("\nTarget (y) Head:")
print(y.head())

Features (X) Head:
      TV  Radio  Newspaper
0  230.1   37.8       69.2
1   44.5   39.3       45.1
2   17.2   45.9       69.3
3  151.5   41.3       58.5
4  180.8   10.8       58.4

Target (y) Head:
0    22.1
1    10.4
2    12.0
3    16.5
4    17.9
Name: Sales, dtype: float64


## Step 4: Check Data Quality & Summary Statistics (Task 3)
We verify that the dataset contains no missing values or duplicate entries, and check the descriptive statistics.

In [4]:
print("Missing Values:")
print(df.isnull().sum())

print(f"\nDuplicate Rows: {df.duplicated().sum()}")

print("\nSummary Statistics:")
print(df.describe())

Missing Values:
TV           0
Radio        0
Newspaper    0
Sales        0
dtype: int64

Duplicate Rows: 0

Summary Statistics:
               TV       Radio   Newspaper       Sales
count  200.000000  200.000000  200.000000  200.000000
mean   147.042500   23.264000   30.554000   15.130500
std     85.854236   14.846809   21.778621    5.283892
min      0.700000    0.000000    0.300000    1.600000
25%     74.375000    9.975000   12.750000   11.000000
50%    149.750000   22.900000   25.750000   16.000000
75%    218.825000   36.525000   45.100000   19.050000
max    296.400000   49.600000  114.000000   27.000000


## Graph 1: Feature Spend vs. Sales Relationships (Exploratory Data Analysis)
Before building the linear regression model, we visualize how each advertising medium relates to sales volume.
This 3-panel scatter plot with trend lines highlights:
- **TV vs Sales:** Strong upward linear trajectory.
- **Radio vs Sales:** Clear positive slope with moderate dispersion.
- **Newspaper vs Sales:** Substantially scattered with a nearly flat trend line.

In [5]:
# Graph 1: 3-Panel Scatter Plot showing Individual Advertising Spend vs Sales
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: TV vs Sales
axes[0].scatter(df['TV'], df['Sales'], color='#1D3557', alpha=0.75, edgecolors='black', linewidth=0.5)
m_tv, b_tv = np.polyfit(df['TV'], df['Sales'], 1)
axes[0].plot(df['TV'], m_tv * df['TV'] + b_tv, color='#E63946', linewidth=2)
axes[0].set_title('TV Spend vs Sales\n(Clear Upward Line: Strong Relationship)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('TV Spend ($k)', fontsize=10)
axes[0].set_ylabel('Sales (k units)', fontsize=10)
axes[0].grid(True, linestyle='--', alpha=0.5)

# Panel 2: Radio vs Sales
axes[1].scatter(df['Radio'], df['Sales'], color='#2A9D8F', alpha=0.75, edgecolors='black', linewidth=0.5)
m_rad, b_rad = np.polyfit(df['Radio'], df['Sales'], 1)
axes[1].plot(df['Radio'], m_rad * df['Radio'] + b_rad, color='#E63946', linewidth=2)
axes[1].set_title('Radio Spend vs Sales\n(Moderate Upward Line: Positive Relationship)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Radio Spend ($k)', fontsize=10)
axes[1].set_ylabel('Sales (k units)', fontsize=10)
axes[1].grid(True, linestyle='--', alpha=0.5)

# Panel 3: Newspaper vs Sales
axes[2].scatter(df['Newspaper'], df['Sales'], color='#F4A261', alpha=0.75, edgecolors='black', linewidth=0.5)
m_news, b_news = np.polyfit(df['Newspaper'], df['Sales'], 1)
axes[2].plot(df['Newspaper'], m_news * df['Newspaper'] + b_news, color='#E63946', linewidth=2)
axes[2].set_title('Newspaper Spend vs Sales\n(Widely Scattered: Very Weak Relationship)', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Newspaper Spend ($k)', fontsize=10)
axes[2].set_ylabel('Sales (k units)', fontsize=10)
axes[2].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Step 5: Split Data into Training and Testing Sets (Task 4)
We partition the dataset into 80% training data (160 samples) and 20% testing data (40 samples), setting `random_state=42` to guarantee reproducibility.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape:  {X_test.shape}")

Training data shape: (160, 3)
Testing data shape:  (40, 3)


## Step 6: Build and Train Multiple Linear Regression Model (Task 5)
We fit the multiple linear regression model on the training data:

$$\text{Sales} = \beta_0 + \beta_1(\text{TV}) + \beta_2(\text{Radio}) + \beta_3(\text{Newspaper})$$

In [7]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


## Step 7: Regression Coefficients & Interpretation (Task 9)
We extract the intercept and coefficients to quantify each channel's contribution.

In [8]:
intercept = model.intercept_
coef_tv = model.coef_[0]
coef_radio = model.coef_[1]
coef_news = model.coef_[2]

print(f"Intercept (β₀): {intercept:.4f}")
print(f"TV coefficient (β₁):        {coef_tv:.4f}")
print(f"Radio coefficient (β₂):     {coef_radio:.4f}")
print(f"Newspaper coefficient (β₃): {coef_news:.4f}")

print(f"\nFitted Regression Equation:")
print(f"Sales = {intercept:.4f} + {coef_tv:.4f}*(TV) + {coef_radio:.4f}*(Radio) + {coef_news:.4f}*(Newspaper)")

Intercept (β₀): 4.7141
TV coefficient (β₁):        0.0545
Radio coefficient (β₂):     0.1009
Newspaper coefficient (β₃): 0.0043

Fitted Regression Equation:
Sales = 4.7141 + 0.0545*(TV) + 0.1009*(Radio) + 0.0043*(Newspaper)


### Coefficient Interpretation:
1. **Baseline Sales (Intercept $\beta_0 = 4.7141$):**
   - If all advertising budgets are zero, the expected baseline organic sales volume is **4.7141k units** (~4,714 units).

2. **Strongest Channel — Radio ($\beta_2 = 0.1009$):**
   - Radio is the **most cost-effective medium per dollar spent**. Each additional **$1,000 invested in Radio** yields an estimated **101 units** (0.1009k units) of sales, holding other channels constant.
   - *TV contributes a substantial volume ($+55$ units per $1k spend), but Radio delivers almost twice the return per dollar invested.*

3. **Weakest Channel — Newspaper ($\beta_3 = 0.0043$):**
   - Newspaper has a **near-zero sales impact**. An additional **$1,000 spend** brings only **4 units** (0.0043k units), making it an ineffective marketing channel in this mix.

## Step 8: Predict Sales for Test Data (Task 6)
We evaluate predictions on the 40 unseen test samples (`X_test`).

In [9]:
y_pred = model.predict(X_test)

results_comparison = pd.DataFrame({
    'Actual Sales': y_test.values,
    'Predicted Sales': np.round(y_pred, 2),
    'Absolute Error': np.round(np.abs(y_test.values - y_pred), 2)
}, index=X_test.index)

print("Sample Predictions vs Actual Sales (First 10 Test Cases):")
print(results_comparison.head(10))

Sample Predictions vs Actual Sales (First 10 Test Cases):
     Actual Sales  Predicted Sales  Absolute Error
95           16.9            17.03            0.13
15           22.4            20.41            1.99
30           21.4            23.72            2.32
158           7.3             9.27            1.97
128          24.7            21.68            3.02
115          12.6            12.57            0.03
69           22.3            21.08            1.22
170           8.4             8.69            0.29
174          16.5            17.24            0.74
45           16.1            16.67            0.57


## Step 9: Evaluate Model Accuracy (Task 8)
We compute key regression metrics on the test set:
- **Mean Absolute Error (MAE):** Average error magnitude.
- **Mean Squared Error (MSE):** Penalizes larger deviations.
- **Root Mean Squared Error (RMSE):** Error in original units (thousands of units).
- **R-Squared ($R^2$) Score:** Proportion of sales variation explained by the model.

In [10]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE:      {mae:.4f}")
print(f"MSE:      {mse:.4f}")
print(f"RMSE:     {rmse:.4f}")
print(f"R2 Score: {r2:.4f}")

MAE:      1.2748
MSE:      2.9078
RMSE:     1.7052
R2 Score: 0.9059


### Evaluation Metrics Summary:
- **$R^2 = 0.9059$ (90.59%):** The three marketing channels account for more than 90% of total variance in sales.
- **$MAE = 1.2748$ k units:** On average, predictions miss actual sales by approximately 1,275 units.
- **$RMSE = 1.7052$ k units:** Low prediction error confirms the model generalizes well to unseen market data.

## Step 10: Predict Sales for Planned Budget Scenario (Task 7)
We forecast sales for the executive team's planned allocation:
- **TV:** $150k (150)
- **Radio:** $20k (20)
- **Newspaper:** $30k (30)

In [11]:
new_budget = pd.DataFrame({
    'TV': [150.0],
    'Radio': [20.0],
    'Newspaper': [30.0]
})

predicted_sales = model.predict(new_budget)[0]

print("Planned Budget:")
print("  TV:        $150,000 (150)")
print("  Radio:     $20,000 (20)")
print("  Newspaper: $30,000 (30)")
print(f"Predicted Sales: {predicted_sales:.4f}k units (~{predicted_sales*1000:,.0f} units)")

Planned Budget:
  TV:        $150,000 (150)
  Radio:     $20,000 (20)
  Newspaper: $30,000 (30)
Predicted Sales: 15.0395k units (~15,040 units)


## Step 11: Model Evaluation Visualizations (Task 10)
We evaluate model predictions on the test set using two diagnostic visualizations:
- **Graph 2: Actual vs. Predicted Scatter Plot** with a diagonal reference line ($y = x$).
- **Graph 3: Sorted Actual vs. Predicted Comparison Bar Chart** in modern Navy & Emerald styling.

In [12]:
# Graph 2: Scatter Plot of Actual vs Predicted Sales with Perfect Fit Reference Line
plt.figure(figsize=(8, 6))

plt.scatter(y_test, y_pred, color='#1D3557', edgecolor='black', s=55, alpha=0.8, label='Test Samples')

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    color='#E63946',
    linestyle='--',
    linewidth=2,
    label='Perfect Fit Line (y = x)'
)

plt.xlabel('Actual Sales (k units)', fontsize=11)
plt.ylabel('Predicted Sales (k units)', fontsize=11)
plt.title('Multiple Linear Regression: Actual vs Predicted Sales', fontsize=12, fontweight='bold')
plt.legend(loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [13]:
# Graph 3: Comparison of Actual vs. Predicted Sales (Sorted Order in Navy & Emerald Palette)
plt.figure(figsize=(13, 6))

# Sort test samples by actual sales for a clean, orderly comparison
sort_order = np.argsort(y_test.values)
actual_sorted = y_test.values[sort_order]
pred_sorted = y_pred[sort_order]

x_positions = np.arange(len(y_test))
bar_w = 0.38

# Custom Colors: Deep Navy Blue for Actual, Emerald Mint for Predicted
plt.bar(
    x_positions - bar_w/2,
    actual_sorted,
    width=bar_w,
    color='#1D3557',  # Deep Navy Blue
    edgecolor='black',
    linewidth=0.5,
    label='Actual Sales'
)
plt.bar(
    x_positions + bar_w/2,
    pred_sorted,
    width=bar_w,
    color='#2A9D8F',  # Emerald Mint Green
    edgecolor='black',
    linewidth=0.5,
    label='Predicted Sales'
)

plt.xlabel('Test Samples (Ranked from Lowest to Highest Actual Sales)', fontsize=11)
plt.ylabel('Sales (k units)', fontsize=11)
plt.title('Actual vs. Predicted Sales Comparison (Sorted Order)', fontsize=13, fontweight='bold')
plt.xticks(np.arange(0, len(y_test), 5), [f'#{i+1}' for i in np.arange(0, len(y_test), 5)])
plt.ylim(0, 27)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend(loc='upper left', frameon=True, fontsize=11)

plt.tight_layout()
plt.show()

### Visual Insights Summary:
1. **Graph 1 (Spend vs. Sales):** Demonstrates that TV and Radio exhibit clear upward linear trends with Sales, whereas Newspaper shows substantial scatter and almost zero slope.
2. **Graph 2 (Actual vs. Predicted Scatter):** Shows points hugging the diagonal red dashed line tightly, confirming high prediction fidelity ($R^2 = 90.6\%$).
3. **Graph 3 (Sorted Bar Chart):** Test samples are sorted by actual sales in ascending order. The emerald predicted bars consistently match the height of the navy actual bars across low, medium, and high sales tiers, confirming that the model has uniform predictive reliability across all sales ranges.

## Step 12: Business Recommendations & Technical Improvements (Task 11)

### 1. Statistical Summary
- **Radio:** $+0.1009$ k sales per $1k spend (highest return on investment per dollar).
- **TV:** $+0.0545$ k sales per $1k spend (major volume driver).
- **Newspaper:** $+0.0043$ k sales per $1k spend (negligible return).

---

### 2. Business Recommendation
- **Action:** Reallocate the proposed $30,000 Newspaper budget into Radio and TV advertising.
- **Impact:** Reinvesting this $30k into Radio will generate an estimated **+3,000 extra units** of product sales without increasing total marketing spend.

---

### 3. Technical Improvements
- **K-Fold Cross-Validation:** Implement 5-fold or 10-fold cross-validation to verify model stability across multiple training folds.
- **Interaction Features:** Introduce interaction terms such as $(\text{TV} \times \text{Radio})$ to model synergies where multi-channel ad exposure amplifies consumer recall.

## Step 13: Executive Summary

| Key Question | Analysis Finding |
| :--- | :--- |
| **Strongest Channel?** | **Radio** by per-dollar efficiency ($+101$ units/\$1k); **TV** by total volume ($+55$ units/\$1k) |
| **Weakest Channel?** | **Newspaper** ($+4$ units/\$1k, negligible return) |
| **Forecast for TV=150, Radio=20, Newspaper=30?** | **15.0395 thousand units** (approx. **15,040 units**) |
| **Model Accuracy on Test Data?** | **$R^2 = 90.59\%$**, **$\text{MAE} = 1.27$ k units**, **$\text{RMSE} = 1.71$ k units** |